In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

folder_path = '/content/drive/MyDrive/MSc AI and Big Data/thesis/data'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Error: Folder '{folder_path}' not found. Please ensure the path is correct and your Drive is mounted.")

Contents of '/content/drive/MyDrive/MSc AI and Big Data/thesis/data':
cia.jpg


In [ ]:
!pip install playwright beautifulsoup4 requests -q
!playwright install chromium --with-deps -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 23.0 MB/s eta 0:00:00
error: unknown option '-q'


In [ ]:
import re
import time
import random
import asyncio
import logging
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────

TOPIC_URLS = [
    "https://www.mathwords.com/topics/middle-school-math/area-perimeter",
    "https://www.mathwords.com/topics/middle-school-math/set-theory",
    "https://www.mathwords.com/topics/middle-school-math/surface-area-volume",
    "https://www.mathwords.com/topics/middle-school-math/fractions-decimals",
    "https://www.mathwords.com/topics/arithmetic/number-types",
    "https://www.mathwords.com/topics/arithmetic/arithmetic-operations",
    "https://www.mathwords.com/topics/arithmetic/factors-multiples",
    "https://www.mathwords.com/topics/arithmetic/measurement-estimation",
]

BASE_URL     = "https://www.mathwords.com"
OUTPUT_ROOT  = "/content/drive/MyDrive/MSc AI and Big Data/thesis/data"

# Set to True for the 3-page smoke test; False for the full run
TEST_MODE    = False
TEST_LIMIT   = 3          # total subpages to process in test mode

# Rate-limit bounds (seconds) — randomised to avoid bot detection
DELAY_MIN    = 2.5
DELAY_MAX    = 5.5

# ── Logging ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Realistic browser headers for requests ───────────────────────────────────

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.mathwords.com/",
    "DNT": "1",
}

In [ ]:
# ── Step 1: Collect subpage links from the "All Terms (X)" section ────────────

def get_subpage_links(topic_url: str) -> list:
    log.info(f"Fetching topic index: {topic_url}")
    resp = requests.get(topic_url, headers=REQUEST_HEADERS, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    target_h2 = None
    for h2 in soup.find_all("h2"):
        if re.match(r"All Terms\s*\(\d+\)", h2.get_text(strip=True)):
            target_h2 = h2
            break

    if not target_h2:
        log.warning(f"  No 'All Terms' h2 on {topic_url}")
        return []

    ul = target_h2.find_next_sibling("ul")
    if not ul:
        log.warning(f"  No <ul> after 'All Terms' h2 on {topic_url}")
        return []

    links = []
    for li in ul.find_all("li"):
        a = li.find("a", href=True)
        if a:
            href = a["href"]
            full = BASE_URL + href if href.startswith("/") else href
            slug = href.strip("/").replace("/", "_").replace(".htm", "")
            links.append({"title": a.get_text(strip=True), "url": full, "slug": slug})

    log.info(f"  {len(links)} subpages found")
    return links

# ── Step 2: Render using print-media emulation ────────────────────────────────

async def render_print_pdf(page, url: str, output_path: str) -> bool:
    """
    Emulates the browser's built-in print dialog (Ctrl+P → Save as PDF).
    The page's own @media print CSS rules govern layout — identical to what
    you get when printing from a real browser manually.
    """
    try:
        # Engage print media BEFORE navigating so the correct CSS is applied
        # from the very first render pass, not retrofitted after the fact.
        await page.emulate_media(media="print")

        await page.goto(url, wait_until="networkidle", timeout=30_000)

        # Give fonts and any deferred assets a moment to settle
        await asyncio.sleep(1.5)

        await page.pdf(
            path=output_path,
            format="A4",
            print_background=True,  # keeps background colours/images
            # Deliberately no custom margin override — let the site's own
            # media print rules (the same ones your browser respects) handle
            # spacing. This is what makes it match the manual print result.
        )
        return True

    except Exception as exc:
        log.error(f"  PDF failed for {url}: {exc}")
        return False

In [ ]:
# ── Step 3: Orchestrate ───────────────────────────────────────────────────────

async def run():
    # Collect all tasks across every topic page
    all_tasks = []
    for topic_url in TOPIC_URLS:
        topic_slug = topic_url.rstrip("/").split("/")[-1]
        for link in get_subpage_links(topic_url):
            all_tasks.append({"topic_slug": topic_slug, **link})
        # Brief pause between index fetches
        time.sleep(random.uniform(1.0, 2.0))

    log.info(f"Total subpages discovered: {len(all_tasks)}")

    if TEST_MODE:
        all_tasks = all_tasks[:TEST_LIMIT]
        log.info(f"TEST MODE — processing first {TEST_LIMIT} subpages only")

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
            ],
        )

        context = await browser.new_context(
            user_agent=REQUEST_HEADERS["User-Agent"],
            locale="en-US",
            extra_http_headers={
                "Accept-Language": "en-US,en;q=0.9",
                "DNT":             "1",
                "Referer":         BASE_URL + "/",
            },
            viewport={"width": 1280, "height": 900},
        )

        page = await context.new_page()

        ok_count   = 0
        fail_count = 0

        for i, task in enumerate(all_tasks, 1):
            out_dir = Path(OUTPUT_ROOT) / task["topic_slug"]
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = str(out_dir / f"{task['slug']}.pdf")

            # Resume-safe: skip already-completed files
            if Path(out_path).exists():
                log.info(f"[{i}/{len(all_tasks)}] SKIP (exists): {task['slug']}")
                ok_count += 1
                continue

            log.info(f"[{i}/{len(all_tasks)}] {task['title']}")
            log.info(f"  URL: {task['url']}")

            success = await render_print_pdf(page, task["url"], out_path)

            if success:
                log.info(f"  ✓ Saved → {out_path}")
                ok_count += 1
            else:
                fail_count += 1

            # Rate-limit pause (skip after the final item)
            if i < len(all_tasks):
                delay = random.uniform(DELAY_MIN, DELAY_MAX)
                log.info(f"  Waiting {delay:.1f}s …")
                await asyncio.sleep(delay)

        await browser.close()

    log.info("─" * 60)
    log.info(f"Done.  ✓ {ok_count} saved   ✗ {fail_count} failed")
    log.info(f"Output root: {OUTPUT_ROOT}")

In [ ]:
#!playwright install
#!playwright install-deps

In [ ]:
await run()